# NDS ΓÇô Migrate Installer Item `definition.json`

Migrates the JSON state (`definition.json`) of a **Microsoft Nonprofit Data Solutions** installer item
(item type `Microsoft.NonprofitData.*`, e.g. `Microsoft.NonprofitData.Fundraising`) into a
**new (open-source) Fabric item** that may have a different item type / name.

## What it does
1. Reads the **source** MS item definition via the Fabric `getDefinition` LRO and extracts the `definition.json` part.
2. Runs a `migrate_schema()` mapping function (identity by default + `schemaVersion`).
3. Writes the mapped JSON into the **target** OSS item via `updateDefinition`, preserving the target's `.platform` part.

## What it does NOT do
- It does **not** move/recreate the actual deployed Fabric artefacts (lakehouses, notebooks, pipelines, semantic model).
  Only the installer item's JSON state (deployment history, selected modules, workspace-move metadata) is migrated.
- `deployedItems[]` keep their original `workspaceId`/`itemId` references.

## Prerequisites
- Run in a Fabric notebook with access to both source and target workspaces.
- The signed-in identity needs **read** on the source item and **write** on the target item.
- The target OSS item must already exist (create it first, then run this notebook).

## 1. Parameters
Find the IDs in the item URL: `.../groups/<WORKSPACE_ID>/.../<ITEM_ID>`.

In [ ]:
# Source = existing Microsoft NDS installer item.
# Leave SRC_WORKSPACE_ID empty to use the workspace this notebook runs in.
# Leave SRC_ITEM_ID empty to auto-discover it by item type (see ITEM_TYPE_FILTER below).
SRC_WORKSPACE_ID = ""  # empty -> current workspace
SRC_ITEM_ID = ""       # empty -> auto-discover

# Substring matched (case-insensitive) against the Fabric item type to recognise the
# installer item. Real types look like "Microsoft.NonprofitData.Fundraising", so the
# "NonprofitData" substring matches them all; narrow it if a workspace has several.
ITEM_TYPE_FILTER = "NonprofitData"

# Target = new open-source installer item (already created, possibly a different item type/name)
DST_WORKSPACE_ID = ""
DST_ITEM_ID = ""

# Safety switch: when True nothing is written, the mapped JSON is only printed.
DRY_RUN = True

# Name of the JSON part that holds the installer state.
DEFINITION_PART_PATH = "definition.json"

FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

## 2. Helpers
`getDefinition` / `updateDefinition` are long-running operations. `FabricRestClient`
(`sempy.fabric`, preinstalled in Fabric notebooks) handles auth and the LRO polling for us
via `lro_wait=True`, so the helpers below stay short.

In [ ]:
import base64
import json

import sempy.fabric as fabric
from notebookutils import mssparkutils

client = fabric.FabricRestClient()  # handles auth + long-running operations


def current_workspace_id():
    ctx = mssparkutils.runtime.context
    return ctx.get("currentWorkspaceId") or ctx.get("workspaceId")


def find_items(workspace_id, type_filter):
    """Items whose Fabric type contains `type_filter` (case-insensitive)."""
    items = client.get(f"/v1/workspaces/{workspace_id}/items").json()["value"]
    needle = type_filter.lower()
    return [i for i in items if needle in (i.get("type") or "").lower()]


def get_parts(workspace_id, item_id):
    """Definition parts [{path, payload, payloadType}] (waits for the LRO)."""
    resp = client.post(
        f"/v1/workspaces/{workspace_id}/items/{item_id}/getDefinition", lro_wait=True
    )
    return resp.json()["definition"]["parts"]


def update_parts(workspace_id, item_id, parts):
    """Write the full parts list back (waits for the LRO)."""
    client.post(
        f"/v1/workspaces/{workspace_id}/items/{item_id}/updateDefinition",
        json={"definition": {"parts": parts}},
        lro_wait=True,
    )


def get_json_part(parts, path):
    part = next((p for p in parts if p.get("path") == path), None)
    return json.loads(base64.b64decode(part["payload"])) if part else None


def set_json_part(parts, path, obj):
    """Replace (or append) a JSON part in-place, preserving the other parts."""
    payload = base64.b64encode(json.dumps(obj, indent=2).encode()).decode()
    for p in parts:
        if p.get("path") == path:
            p["payload"], p["payloadType"] = payload, "InlineBase64"
            return parts
    parts.append({"path": path, "payload": payload, "payloadType": "InlineBase64"})
    return parts


print("Helpers ready.")


## 3. Resolve the source item
If `SRC_WORKSPACE_ID` / `SRC_ITEM_ID` are empty, use the current workspace and auto-discover
the Microsoft NDS installer item by its item type (`ITEM_TYPE_FILTER`). If more than one
candidate is found, the IDs are printed so you can set `SRC_ITEM_ID` explicitly.

In [ ]:
if not SRC_WORKSPACE_ID:
    SRC_WORKSPACE_ID = current_workspace_id()

if not SRC_ITEM_ID:
    candidates = find_items(SRC_WORKSPACE_ID, ITEM_TYPE_FILTER)
    for c in candidates:
        print(f"  found: {c['displayName']!r}  type={c['type']}  id={c['id']}")
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly 1 item matching '{ITEM_TYPE_FILTER}', found {len(candidates)}. "
            "Set SRC_ITEM_ID explicitly."
        )
    SRC_ITEM_ID = candidates[0]["id"]

print("Source:", SRC_WORKSPACE_ID, "/", SRC_ITEM_ID)


## 4. Read the source installer state

In [ ]:
src_parts = get_parts(SRC_WORKSPACE_ID, SRC_ITEM_ID)
src_definition = get_json_part(src_parts, DEFINITION_PART_PATH)
assert src_definition is not None, f"Source item has no '{DEFINITION_PART_PATH}' part."

print("Parts:", [p["path"] for p in src_parts])
print("Deployments:", len(src_definition.get("deployments", [])))
print(json.dumps(src_definition, indent=2)[:2000])


## 5. Map the schema
Keep this as **identity** if the OSS item reuses the same `definition.json` shape
(`deployments`, `oneLakePackages`, `workspaceMove`). Extend `migrate_schema()` when the OSS
schema diverges (rename keys, drop fields, change package IDs, etc.).

In [ ]:
OSS_SCHEMA_VERSION = 1


def migrate_schema(source: dict) -> dict:
    """Map the Microsoft installer definition.json into the OSS schema.
    Default = identity passthrough + schemaVersion. Add remapping below when needed.
    """
    target = {
        "schemaVersion": OSS_SCHEMA_VERSION,
        "deployments": source.get("deployments", []),
        "oneLakePackages": source.get("oneLakePackages", []),
    }
    if "workspaceMove" in source:
        target["workspaceMove"] = source["workspaceMove"]
    return target


target_definition = migrate_schema(src_definition)
print(json.dumps(target_definition, indent=2)[:2000])


## 6. Write into the target OSS item
Reads the target's current parts, replaces only `definition.json`, and keeps `.platform`
(item type / display name) untouched. Set `DRY_RUN = False` to actually write.

In [ ]:
assert DST_WORKSPACE_ID and DST_ITEM_ID, "Set DST_WORKSPACE_ID and DST_ITEM_ID first."

dst_parts = get_parts(DST_WORKSPACE_ID, DST_ITEM_ID)  # keeps .platform etc.
set_json_part(dst_parts, DEFINITION_PART_PATH, target_definition)

if DRY_RUN:
    print("DRY_RUN = True -> nothing written. Parts that WOULD be sent:", [p["path"] for p in dst_parts])
else:
    update_parts(DST_WORKSPACE_ID, DST_ITEM_ID, dst_parts)
    print("Migration written to target item.")


## 7. Verify
Re-read the target `definition.json` and confirm the migrated content.

In [ ]:
if not DRY_RUN:
    verify = get_json_part(get_parts(DST_WORKSPACE_ID, DST_ITEM_ID), DEFINITION_PART_PATH)
    print("schemaVersion:", verify.get("schemaVersion"))
    print("deployments:", len(verify.get("deployments", [])))
else:
    print("DRY_RUN was True - skipping verification.")
